# Lab 3 — Lemma/POS Baseline

**Dataset:** 20 Newsgroups — `alt.atheism` / `sci.electronics` / `soc.religion.christian`
**Task:** A — Text Classification
**Tool:** Stanza (English model)

| Section | Content |
|---|---|
| 0 | Setup |
| 1 | Load processed_v2 |
| 2 | Lemma/POS extraction |
| 3 | Baseline comparison (TF-IDF + LR) |
| 4 | Error analysis |
| 5 | Conclusion |
| 6 | Generate audit_summary_lab3.md |

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'stanza', 'scikit-learn', 'tqdm', 'pandas'])

0

In [2]:
# stanza must be imported before sys.path gets a non-ASCII entry (Windows DLL quirk)
import stanza

import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not Path(f'/content/NLP-Lab-works').exists():
        os.system(f'git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git')
    REPO_ROOT = Path(f'/content/NLP-Lab-works')
else:
    _p = Path.cwd().resolve()
    REPO_ROOT = None
    for _ in range(6):
        if (_p / 'src' / 'preprocess.py').exists():
            REPO_ROOT = _p
            break
        _p = _p.parent
    if REPO_ROOT is None:
        raise FileNotFoundError(f'Cannot locate repo root from {Path.cwd()}.')

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Repo root:', REPO_ROOT)

Repo root: C:\Users\Я\OneDrive\Рабочий стол\lpnu lab\5 curs\обробка мови\lab1


In [3]:
import json, warnings
from datetime import date
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from ling_features import get_pipeline, extract_features, process_dataframe

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
print('Imports OK')

Imports OK


## 1. Load processed_v2

In [4]:
v2_path = Path('data/processed_v2/processed_v2.csv')
df = pd.read_csv(v2_path)
df['text_v2'] = df['text_v2'].fillna('')
print(f'Loaded {len(df):,} rows  |  columns: {list(df.columns)}')
print(df['category'].value_counts().to_string())
df[['id', 'category', 'text_v2']].head(3)

Loaded 6,383 rows  |  columns: ['id', 'text_v2', 'sentence_count', 'char_length', 'word_count', 'label_id', 'category', 'subject', 'n_urls', 'n_emails', 'n_phones', 'n_quote_lines']
category
alt.atheism               2405
soc.religion.christian    2003
sci.electronics           1975


,id,category,text_v2
0,0,alt.atheism,"Oops, sorry, my words, not the words of the Qur'an.\n\nNote that ""(the celestial bodies)"" in the..."
1,1,soc.religion.christian,"Though there is a command in the law not to heed to one who prophecies\nfalsely, it is still pos..."
2,2,soc.religion.christian,"I haven't followed whatever discussion there may have been on these\npeople, but I feel that C. ..."


## 2. Lemma/POS Extraction

In [5]:
stanza.download('en', verbose=False)
nlp = get_pipeline('en')
print('Stanza pipeline ready.')

sample = df['text_v2'].iloc[0]
result = extract_features(sample, nlp)
print('Input  (first 200):', sample[:200])
print('Lemma  (first 200):', result['lemma_text'][:200])
print('POS    (first 20) :', ' '.join(result['pos_seq'].split()[:20]))
print('Content(first 15) :', ' '.join(result['content_lemma_text'].split()[:15]))

Stanza pipeline ready.
Input  (first 200): Oops, sorry, my words, not the words of the Qur'an.

Note that "(the celestial bodies)" in the above verse is an
interpolation (which is why it is in brackets) -- it is the translator's
(incorrect, IM
Lemma  (first 200): oops , sorry , my word , not the word of the qur'an . note that " ( the celestial body ) " in the above verse be a interpolation ( which be why it be in bracket ) -- it be the translator 's ( incorrec
POS    (first 20) : INTJ PUNCT INTJ PUNCT PRON NOUN PUNCT PART DET NOUN ADP DET PROPN PUNCT VERB SCONJ PUNCT PUNCT DET ADJ
Content(first 15) : word word note celestial body above verse interpolation bracket translator incorrect interpretation translation study research


In [ ]:
proc_v3_path = Path('data/processed_v3/processed_v3.csv')

if proc_v3_path.exists():
    print(f'Loading cached {proc_v3_path}...')
    df_v3 = pd.read_csv(proc_v3_path)
    for col in ('lemma_text', 'pos_seq', 'content_lemma_text'):
        df_v3[col] = df_v3[col].fillna('')
else:
    print(f'Extracting features for {len(df):,} documents (first run is slow)...')
    ling_rows = process_dataframe(df, text_col='text_v2', nlp=nlp)
    df_v3 = df.copy()
    df_v3['lemma_text']         = [r['lemma_text']         for r in ling_rows]
    df_v3['pos_seq']            = [r['pos_seq']            for r in ling_rows]
    df_v3['content_lemma_text'] = [r['content_lemma_text'] for r in ling_rows]
    df_v3.to_csv(proc_v3_path, index=False)
    print(f'Saved: {proc_v3_path}')

print('Sample lemma_text:', df_v3['lemma_text'].iloc[0][:300])

Extracting features for 6,383 documents (first run is slow)...


Extracting lemma/POS: 100%|██████████| 6383/6383 [9:59:45<00:00,  5.64s/doc]       


Saved: data\processed_v2\processed_v3.csv
Sample lemma_text: oops , sorry , my word , not the word of the qur'an . note that " ( the celestial body ) " in the above verse be a interpolation ( which be why it be in bracket ) -- it be the translator 's ( incorrect , imho ) interpretation . here be maurice bucaille 's translation ( he study arabic for his resear


In [7]:
for i in range(5):
    row = df_v3.iloc[i]
    print(f'--- doc {i} | {row["category"]} ---')
    print(f'  RAW    : {row["text_v2"][:180]}')
    print(f'  LEMMA  : {row["lemma_text"][:180]}')
    print(f'  CONTENT: {row["content_lemma_text"][:180]}')
    print()

--- doc 0 | alt.atheism ---
  RAW    : Oops, sorry, my words, not the words of the Qur'an.

Note that "(the celestial bodies)" in the above verse is an
interpolation (which is why it is in brackets) -- it is the transla
  LEMMA  : oops , sorry , my word , not the word of the qur'an . note that " ( the celestial body ) " in the above verse be a interpolation ( which be why it be in bracket ) -- it be the tran
  CONTENT: word word note celestial body above verse interpolation bracket translator incorrect interpretation translation study research science verse create night day sun moon travel orbit 

--- doc 1 | soc.religion.christian ---
  RAW    : Though there is a command in the law not to heed to one who prophecies
falsely, it is still possible for the one who has prophecied falsely
to prophecy truely again. Take, for exam
  LEMMA  : though there be a command in the law not to heed to one who prophecy falsely , it be still possible for the one who have prophecy falsely to prophecy tru

## 3. Baseline Comparison

In [8]:
X_raw     = df_v3['text_v2'].tolist()
X_lemma   = df_v3['lemma_text'].tolist()
X_content = df_v3['content_lemma_text'].tolist()
y         = df_v3['label_id'].tolist()

# Single index split keeps all three feature sets perfectly aligned
indices = list(range(len(df_v3)))
tr_idx, te_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

X_raw_tr  = [X_raw[i]     for i in tr_idx];  X_raw_te  = [X_raw[i]     for i in te_idx]
X_lem_tr  = [X_lemma[i]   for i in tr_idx];  X_lem_te  = [X_lemma[i]   for i in te_idx]
X_con_tr  = [X_content[i] for i in tr_idx];  X_con_te  = [X_content[i] for i in te_idx]
y_tr      = [y[i]         for i in tr_idx];  y_te      = [y[i]         for i in te_idx]

print(f'Train: {len(y_tr):,}  |  Test: {len(y_te):,}')

Train: 5,106  |  Test: 1,277


In [9]:
def run_baseline(X_train, X_test, y_train, y_test, label):
    tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)
    clf   = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(tfidf.fit_transform(X_train), y_train)
    preds = clf.predict(tfidf.transform(X_test))
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average='macro')
    print(f'[{label}]  Acc={acc:.4f}  Macro-F1={f1:.4f}')
    return {'label': label, 'accuracy': acc, 'macro_f1': f1, 'preds': preds}

In [10]:
res1 = run_baseline(X_raw_tr,  X_raw_te,  y_tr, y_te, 'Baseline 1: processed_v2')
res2 = run_baseline(X_lem_tr,  X_lem_te,  y_tr, y_te, 'Baseline 2: lemma_text')
res3 = run_baseline(X_con_tr,  X_con_te,  y_tr, y_te, 'Baseline 3: content_lemma')

[Baseline 1: processed_v2]  Acc=0.9843  Macro-F1=0.9846
[Baseline 2: lemma_text]  Acc=0.9812  Macro-F1=0.9815
[Baseline 3: content_lemma]  Acc=0.9640  Macro-F1=0.9643


In [11]:
results_df = pd.DataFrame([
    {'Approach': r['label'], 'Accuracy': f"{r['accuracy']:.4f}", 'Macro-F1': f"{r['macro_f1']:.4f}"}
    for r in [res1, res2, res3]
])
print(results_df.to_string(index=False))
print(f"\nLemma vs Raw: Acc delta={res2['accuracy']-res1['accuracy']:+.4f}"
      f"  F1 delta={res2['macro_f1']-res1['macro_f1']:+.4f}")

                 Approach Accuracy Macro-F1
 Baseline 1: processed_v2   0.9843   0.9846
   Baseline 2: lemma_text   0.9812   0.9815
Baseline 3: content_lemma   0.9640   0.9643

Lemma vs Raw: Acc delta=-0.0031  F1 delta=-0.0031


In [12]:
target_names = ['alt.atheism', 'sci.electronics', 'soc.religion.christian']
print('=== Baseline 1: processed_v2 ===')
print(classification_report(y_te, res1['preds'], target_names=target_names))
print('=== Baseline 2: lemma_text ===')
print(classification_report(y_te, res2['preds'], target_names=target_names))

=== Baseline 1: processed_v2 ===
                        precision    recall  f1-score   support

           alt.atheism       0.97      1.00      0.98       481
       sci.electronics       1.00      0.98      0.99       395
soc.religion.christian       0.99      0.97      0.98       401

              accuracy                           0.98      1277
             macro avg       0.99      0.98      0.98      1277
          weighted avg       0.98      0.98      0.98      1277

=== Baseline 2: lemma_text ===
                        precision    recall  f1-score   support

           alt.atheism       0.97      0.99      0.98       481
       sci.electronics       0.99      0.98      0.99       395
soc.religion.christian       0.99      0.97      0.98       401

              accuracy                           0.98      1277
             macro avg       0.98      0.98      0.98      1277
          weighted avg       0.98      0.98      0.98      1277



## 4. Error Analysis — Lemmatization Issues

In [13]:
edge_path  = Path('tests/ling_edge_cases.jsonl')
edge_cases = [json.loads(l) for l in edge_path.read_text(encoding='utf-8').splitlines() if l.strip()]
print(f'Loaded {len(edge_cases)} edge cases')

Loaded 20 edge cases


In [14]:
rows = []
for ec in edge_cases:
    feat = extract_features(ec['raw_text'], nlp)
    if ec['category'] == 'pii_token':
        orig   = [t for t in ('<URL>', '<EMAIL>', '<PHONE>') if t in ec['raw_text']]
        pii_ok = all(t in feat['lemma_text'] for t in orig)
    else:
        pii_ok = None
    rows.append({
        'id':       ec['id'],
        'category': ec['category'],
        'raw':      ec['raw_text'][:55],
        'lemma':    feat['lemma_text'][:75],
        'pos_10':   ' '.join(feat['pos_seq'].split()[:10]),
        'pii_ok':   pii_ok,
    })

print(pd.DataFrame(rows).to_string(index=False))

 id       category                                               raw                                             lemma                                    pos_10 pii_ok
  1   abbreviation                  BTW, I disagree with your point.                btw , i disagree with your point .  INTJ PUNCT PRON VERB ADP PRON NOUN PUNCT   None
  2      pii_token                Contact me at <EMAIL> for details.                 contact i at <EMAIL> for detail .         VERB PRON ADP NOUN ADP NOUN PUNCT   True
  3     hyphenated                  The follow-up message was clear.                the follow - up message be clear .     DET NOUN PUNCT ADP NOUN AUX ADJ PUNCT   None
  4     possessive                          God's grace is infinite.                        god 's grace be infinite .             PROPN PART NOUN AUX ADJ PUNCT   None
  5        archaic                              Thou shalt not kill.                             thou shalt not kill .                  PRON AUX PART VERB PUNCT

In [15]:
focus = ['pii_token', 'abbreviation', 'contraction', 'hyphenated', 'proper_noun']
for ec in [e for e in edge_cases if e['category'] in focus][:10]:
    feat = extract_features(ec['raw_text'], nlp)
    print(f"id={ec['id']}  [{ec['category']}]")
    print(f"  RAW     : {ec['raw_text']}")
    print(f"  LEMMA   : {feat['lemma_text']}")
    print(f"  POS     : {' '.join(feat['pos_seq'].split()[:15])}")
    print(f"  EXPECTED: {ec['expected_behavior']}")
    print()

id=1  [abbreviation]
  RAW     : BTW, I disagree with your point.
  LEMMA   : btw , i disagree with your point .
  POS     : INTJ PUNCT PRON VERB ADP PRON NOUN PUNCT
  EXPECTED: BTW should be tagged as NOUN or PROPN and its lemma should stay 'btw', not decomposed into garbage tokens

id=2  [pii_token]
  RAW     : Contact me at <EMAIL> for details.
  LEMMA   : contact i at <EMAIL> for detail .
  POS     : VERB PRON ADP NOUN ADP NOUN PUNCT
  EXPECTED: <EMAIL> token must be preserved unchanged in lemma_text output after PII protect/restore cycle

id=3  [hyphenated]
  RAW     : The follow-up message was clear.
  LEMMA   : the follow - up message be clear .
  POS     : DET NOUN PUNCT ADP NOUN AUX ADJ PUNCT
  EXPECTED: follow-up may be split into 'follow', '-', 'up' or kept as one token; lemma should be 'follow-up' or 'follow up', not dropped

id=6  [proper_noun]
  RAW     : Jesus Christ was born in Bethlehem.
  LEMMA   : jesus christ be bear in bethlehem .
  POS     : PROPN PROPN AUX VERB A

## 5. Conclusion

In [16]:
delta_f1  = res2['macro_f1'] - res1['macro_f1']
direction = 'improved' if delta_f1 > 0 else 'did not improve'

conclusion = [
    f'Lemmatization {direction} classification (Macro-F1 delta={delta_f1:+.4f}).',
    '',
    'Why improvement is small on this dataset:',
    '  - 3 topically distinct classes -> vocabularies barely overlap ->',
    '    TF-IDF captures discriminative signal even from inflected forms.',
    "  - Lemmatization can merge forms that carry class signal",
    "    ('electronics'/'electronic' both point to sci.electronics).",
    '',
    'Content-word filtering (Baseline 3) hurt because function words',
    "carry class signal here: 'thou', 'thy', 'shalt' -> soc.religion.christian.",
    '',
    'When lemmas HELP:',
    "  - Retrieval: 'run'/'ran'/'running' all match the same query.",
    '  - Clustering: reduces vocabulary sparsity.',
    '  - Rich-morphology languages benefit more than English.',
    '',
    'Lemmatization errors found in edge cases:',
    '  - pii_token    : <URL>/<EMAIL>/<PHONE> survive unchanged',
    '  - abbreviation : BTW/RAM/IMHO tagged as NOUN/PROPN (incorrect)',
    "  - contraction  : don't split correctly by Stanza",
    "  - proper_noun  : 'alt.atheism' tokenized across dots (expected)",
    '  - internet_slang: AFAIK/IMHO get unexpected POS tags',
    '',
    'Decision: processed_v2 for classification | lemma_text for retrieval/clustering.',
]
print('\n'.join(conclusion))

Lemmatization did not improve classification (Macro-F1 delta=-0.0031).

Why improvement is small on this dataset:
  - 3 topically distinct classes -> vocabularies barely overlap ->
    TF-IDF captures discriminative signal even from inflected forms.
  - Lemmatization can merge forms that carry class signal
    ('electronics'/'electronic' both point to sci.electronics).

Content-word filtering (Baseline 3) hurt because function words
carry class signal here: 'thou', 'thy', 'shalt' -> soc.religion.christian.

When lemmas HELP:
  - Retrieval: 'run'/'ran'/'running' all match the same query.
  - Clustering: reduces vocabulary sparsity.
  - Rich-morphology languages benefit more than English.

Lemmatization errors found in edge cases:
  - pii_token    : <URL>/<EMAIL>/<PHONE> survive unchanged
  - abbreviation : BTW/RAM/IMHO tagged as NOUN/PROPN (incorrect)
  - contraction  : don't split correctly by Stanza
  - proper_noun  : 'alt.atheism' tokenized across dots (expected)
  - internet_slang: 

## 6. Generate audit_summary_lab3.md

In [ ]:
b1_acc, b1_f1 = res1['accuracy'], res1['macro_f1']
b2_acc, b2_f1 = res2['accuracy'], res2['macro_f1']
b3_acc, b3_f1 = res3['accuracy'], res3['macro_f1']
verdict = 'improved' if b2_f1 > b1_f1 else 'did not improve'
d_acc   = b2_acc - b1_acc
d_f1    = b2_f1  - b1_f1

lines = [
    '# Audit Summary — Lab 3 Lemma/POS Baseline',
    '',
    f'**Generated:** {date.today()}  ',
    '**Dataset:** 20 Newsgroups (alt.atheism / sci.electronics / soc.religion.christian)  ',
    '**Task:** A — Text Classification  ',
    '**Tool:** Stanza (English model)  ',
    f'**Total documents:** {len(df_v3):,}  ',
    '**Stanza truncation:** 300 words per document  ',
    '',
    '## Baseline Results',
    '',
    '| Approach | Accuracy | Macro-F1 |',
    '|---|---|---|',
    f'| Baseline 1: processed_v2 | {b1_acc:.4f} | {b1_f1:.4f} |',
    f'| Baseline 2: lemma_text | {b2_acc:.4f} | {b2_f1:.4f} |',
    f'| Baseline 3: content lemmas (NOUN/ADJ/VERB) | {b3_acc:.4f} | {b3_f1:.4f} |',
    '',
    f'**Lemma vs Raw:** Accuracy delta={d_acc:+.4f} | Macro-F1 delta={d_f1:+.4f}',
    '',
    '## Error Analysis',
    '',
    f'- Edge cases tested: {len(edge_cases)}  ',
    '- Source: `tests/ling_edge_cases.jsonl`  ',
    '- PII tokens survive lemmatization: Yes  ',
    '- Main issues: abbreviations, internet slang, proper nouns with dots  ',
    '',
    '## Conclusion',
    '',
    f'Lemmatization {verdict} classification (Macro-F1 delta={d_f1:+.4f}).',
    'Topically distinct classes provide strong TF-IDF signal without normalization.',
    'Lemmas recommended for retrieval and clustering tasks.',
    '',
    '**Decision:** `processed_v2` for classification · `lemma_text` for retrieval/clustering.',
    '',
    '## Files',
    '',
    '- `data/processed_v3/processed_v3.csv` — lemma_text, pos_seq, content_lemma_text  ',
    '- `data/sample/sample_v3.csv` — 100-row sample  ',
]

Path('docs/audit_summary_lab3.md').write_text('\n'.join(lines), encoding='utf-8')
print('Written: docs/audit_summary_lab3.md')

Written: docs/audit_summary_lab3.md


In [18]:
Path('data/sample').mkdir(parents=True, exist_ok=True)
df_v3.head(100).to_csv('data/sample/sample_v3.csv', index=False)

print('=' * 50)
print('  Lab 3 COMPLETE')
print('=' * 50)
print(f'  Baseline 1 (raw):     Acc={b1_acc:.4f}  F1={b1_f1:.4f}')
print(f'  Baseline 2 (lemma):   Acc={b2_acc:.4f}  F1={b2_f1:.4f}')
print(f'  Baseline 3 (content): Acc={b3_acc:.4f}  F1={b3_f1:.4f}')
print(f'  Edge cases : {len(edge_cases)}')
print(f'  Audit      : docs/audit_summary_lab3.md')
print(f'  Sample     : data/sample/sample_v3.csv')

  Lab 3 COMPLETE
  Baseline 1 (raw):     Acc=0.9843  F1=0.9846
  Baseline 2 (lemma):   Acc=0.9812  F1=0.9815
  Baseline 3 (content): Acc=0.9640  F1=0.9643
  Edge cases : 20
  Audit      : docs/audit_summary_lab3.md
  Sample     : data/sample/sample_v3.csv
